# PySpark 7: monitoring, performance and tuning

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-07-monitoring-and-performance.ipynb)

An orders job that used to finish before the reporting deadline has slowed down. A larger cluster may help
one symptom while leaving the real bottleneck untouched. Start by finding what changed in the plan and the
stage metrics.

This module connects physical plans with runtime evidence, then works through the levers that matter:
shuffles and partition counts, adaptive execution, skew and salting, join strategies, file layout and
pruning, bucketing and statistics, caching and checkpoints, memory and executor sizing, and Python UDFs. It
ends with arithmetic exercises, real incidents and an interview checklist.

Each section explains how the feature works, where it matters and what to watch out for, then ends with a runnable example. Run it, read the output, and compare it with the notes on what to notice. Run the cells in order.

**Prerequisites:** [PySpark 3: joins and aggregations](pyspark-03-joins-and-aggregations.ipynb) and [PySpark 5: writing, pipelines and quality](pyspark-05-writing-pipelines-and-quality.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The session below also writes an event log, which the monitoring section reads back. The small tables from
[PySpark 1](pyspark-01-foundations.ipynb#sample-data-used-in-every-module) are too small to show a shuffle or
a straggler, so this module builds a larger pair of tables instead.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write
EVENT_LOG_DIR = os.path.join(WORK_DIR, "event-log")  # read back in the monitoring section
os.makedirs(EVENT_LOG_DIR)

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", EVENT_LOG_DIR)
    .config("spark.eventLog.compress", "false")  # plain JSON lines, readable below
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date

from pyspark.sql import functions as F

import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

`big_orders` has 200,000 rows, and customer `0` owns 40 percent of them: the "guest checkout" key that makes
real joins skew. `big_customers` is a 5,000-row dimension. The cell also resets the session to a static
baseline (16 shuffle partitions, AQE off), so each lever can be switched on deliberately.

In [ ]:
# Tuning lessons start from a static baseline; later cells switch levers on one at a time.
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "false")

from pyspark.sql.functions import spark_partition_id


def partition_sizes(df):
    """Row count per partition, in partition order: the skew signal the Stages tab shows."""
    return [r["count"] for r in df.groupBy(spark_partition_id().alias("p")).count().orderBy("p").collect()]

ROWS, CUSTOMERS, HOT_SHARE = 200_000, 5_000, 0.4
big_orders = spark.range(ROWS).select(
    F.col("id").alias("order_id"),
    F.when(F.col("id") % 10 < HOT_SHARE * 10, F.lit(0)).otherwise(F.col("id") % CUSTOMERS).alias("customer_id"),
    F.date_add(F.lit("2026-10-01").cast("date"), (F.col("id") % 30).cast("int")).alias("order_date"),
    (F.col("id") % 997 / 10).cast("decimal(12,2)").alias("amount"),
    F.element_at(F.array(F.lit("paid"), F.lit("shipped"), F.lit("placed")), (F.col("id") % 3 + 1).cast("int")).alias("status"),
)
big_customers = spark.range(CUSTOMERS).select(
    F.col("id").alias("customer_id"),
    F.concat(F.lit("segment-"), (F.col("id") % 4).cast("string")).alias("customer_segment"),
)

# A partitioned Parquet copy, so scans, pruning and pushdown behave like a real table
ORDERS_PATH = os.path.join(WORK_DIR, "big_orders")
big_orders.write.mode("overwrite").partitionBy("order_date").parquet(ORDERS_PATH)
big_orders = spark.read.parquet(ORDERS_PATH)

hot = big_orders.where("customer_id = 0").count()
print(f"rows: {big_orders.count():,}  hot-key rows: {hot:,}")

## Read a physical plan

Read `explain("formatted")` bottom-up and look for:

- `Scan parquet` with `ReadSchema`: which columns are read;
- `PartitionFilters`: directories pruned at plan time; `PushedFilters`: predicates the file reader evaluates;
- `Exchange`: a shuffle, and therefore a stage boundary;
- `BroadcastHashJoin` versus `SortMergeJoin`: whether the join shuffled both sides;
- `AdaptiveSparkPlan isFinalPlan=false`: AQE may still rewrite the plan at runtime.

Catalyst runs analysis (resolve names and types), logical optimization (pushdown, constant folding, column
pruning), physical planning (pick the cheapest candidate from statistics), and whole-stage code generation
(operators fused into one generated function, shown as `WholeStageCodegen` or `*`). `explain("cost")` prints
the size and row estimates the planner used.

**Where it matters**
- **Before tuning anything:** the plan tells you whether the problem is a scan, a shuffle or a join.
- **Code review:** an unexpected `Exchange` or a missing `PartitionFilters` is visible before the job runs.

**Watch out for**
- Treating the root of the plan as the expensive part. Cost sits in the scans, exchanges and joins.
- A pre-execution AQE plan is a guess. Confirm with the final plan or the stage metrics.

### Example

Filter by date, join to customers and aggregate by segment. The plan shows partition pruning, a broadcast join and one exchange; the cost plan shows the estimates.

In [ ]:
result = (
    big_orders
    .where(F.col("order_date") >= F.lit("2026-10-20"))
    .join(big_customers, "customer_id")
    .groupBy("customer_segment")
    .agg(F.sum("amount").alias("revenue"))
)
plan = plan_of(result)
print(plan)

In [ ]:
cost_plan = plan_of(result, "cost")
print(cost_plan[:1500])

## Monitor jobs with the Spark UI and event logs

The Spark UI lives on the driver (port 4040 while the application runs) and in the history server
afterwards. Each tab answers a question:

- **Stages:** sort by duration, open the slowest stage, compare the maximum task duration with the median, and
  read shuffle read and write and spill.
- **SQL / DataFrame:** the executed plan with row counts per operator, and whether AQE changed it.
- **Executors:** GC time, failed tasks, lost executors, and how evenly work is spread.
- **Storage:** which cached frames are materialized and how much memory they hold.

| Metric you see | Likely cause | Fix |
| --- | --- | --- |
| Max task duration much greater than the median | Skew or one oversized partition | AQE skew join, salt the hot key |
| Large shuffle read or write | A wide transformation | Broadcast the small side, pre-aggregate, filter earlier |
| Spill (memory) and spill (disk) above zero | Partition does not fit execution memory | More shuffle partitions, then more memory |
| GC time a large fraction of task time | Too many live Java objects | Smaller caches, serialized storage, smaller executors |
| Many tiny tasks, idle CPU | Over-partitioning | Fewer partitions, AQE coalescing |
| Tiny input tasks and slow listing | Small files | Compact, tune write layout |
| Executor lost or container killed | Overhead memory exhausted | Raise `spark.executor.memoryOverhead`, fewer cores per executor |
| Driver out of memory | `collect`, `toPandas` or a large broadcast | Aggregate in the cluster, lower the broadcast threshold |

This notebook turns the UI off to stay quiet in Colab, so the cells read the same evidence from plans,
partition counts and the event log. Event logs are JSON lines, one event per line; production jobs set
`spark.eventLog.enabled=true` and `spark.eventLog.dir` so the history server can show finished applications.

**Where it matters**
- **Slow-job investigations:** the stage table and the metric table above name the class of problem.
- **Regression tracking:** per-stage shuffle and spill bytes from event logs, compared run to run.

**Watch out for**
- Raising executor memory because a stage is slow. Diagnose the class first: spill, skew, shuffle size or tiny tasks each have a different lever.
- `explain` is a plan, not a runtime fact. Only the UI and the event log show what actually happened.

### Example: read stage metrics from the event log

Run one shuffle, then read this session's event log and sum shuffle and spill bytes per stage.

In [ ]:
import glob
import json
import os
from collections import defaultdict


def event_log_files(log_path):
    """Spark 3 writes one event log file; Spark 4 writes a directory of rolling event files."""
    if os.path.isdir(log_path):
        return sorted(glob.glob(os.path.join(log_path, "events_*")))
    return [log_path]


def stage_metrics(log_path):
    """Sum tasks, shuffle and spill bytes per stage from a Spark event log."""
    totals = defaultdict(lambda: {"tasks": 0, "shuffle_read": 0, "shuffle_write": 0, "spill_disk": 0, "spill_mem": 0})
    for path in event_log_files(log_path):
        with open(path) as handle:
            for line in handle:
                event = json.loads(line)
                if event.get("Event") != "SparkListenerTaskEnd":
                    continue
                metrics = event.get("Task Metrics") or {}
                read = metrics.get("Shuffle Read Metrics") or {}
                write = metrics.get("Shuffle Write Metrics") or {}
                stage = totals[event["Stage ID"]]
                stage["tasks"] += 1
                stage["shuffle_read"] += read.get("Remote Bytes Read", 0) + read.get("Local Bytes Read", 0)
                stage["shuffle_write"] += write.get("Shuffle Bytes Written", 0)
                stage["spill_disk"] += metrics.get("Disk Bytes Spilled", 0)
                stage["spill_mem"] += metrics.get("Memory Bytes Spilled", 0)
    return dict(totals)

spark.conf.set("spark.sql.adaptive.enabled", "false")
big_orders.groupBy("customer_id").agg(F.sum("amount")).collect()  # one shuffle to measure

(log_path,) = glob.glob(os.path.join(EVENT_LOG_DIR, "*"))
metrics = stage_metrics(log_path)
heaviest = max(metrics.items(), key=lambda item: item[1]["shuffle_write"])
print(f"{len(metrics)} stages in the log; heaviest shuffle writer: stage {heaviest[0]} {heaviest[1]}")

### Example: log row counts at a pipeline boundary

Row counts are the cheapest monitoring signal: a step that suddenly drops or multiplies rows is a correctness
bug before it is a performance one. Log input and output counts for a step into a metrics table.

In [ ]:
input_rows = big_orders.count()
paid = big_orders.where("status = 'paid'")
output_rows = paid.count()

spark.createDataFrame(
    [("orders_paid", input_rows, output_rows)], "step string, input_rows long, output_rows long"
).write.mode("append").saveAsTable("ops_pipeline_metrics")

logged = spark.table("ops_pipeline_metrics").first()
print(logged.asDict())

## Treat tuning as a measured experiment

Start with evidence, not configuration: reproduce the slow job on representative data, read the UI and the
plan, name the bottleneck (scan, shuffle, skew, spill, join, small files, driver memory or the sink), change
one thing, and measure again on the same data, code and cluster.

A stage finishes only when its slowest task does. A stage with 1,000 balanced tasks and one straggler is as
slow as the straggler, however much memory you add. Record the same fields before and after: wall clock,
shuffle read and write, spill, task count, maximum task duration, GC time, output file count and cost per
run. Shuffle bytes are the leading indicator; wall clock is what the business sees but also the noisiest, so
compare medians over several runs.

**Where it matters**
- **Proving a change:** "shuffle fell from 1.2 TB to 20 GB and spill to zero" is evidence; "I enabled AQE" is not.
- **Cost reviews:** halving runtime on a cluster twice the size buys latency, not efficiency.

**Watch out for**
- Benchmarking with `count()`. The optimizer can skip columns and work that your real write needs; measure the action you want to improve.
- Comparing runs with different cache states, or changing two settings at once.
- Forgetting to check that the output is identical after the change.

### Example

Compare a baseline and a tuned run field by field.

In [ ]:
import json

# Record the same fields before and after, then compare like with like.
def summarize(before, after):
    def delta(key):
        b, a = before[key], after[key]
        change = (a - b) / b * 100 if b else 0.0
        return f"{b:,} -> {a:,} ({change:+.1f}%)"

    for key in ("wall_clock_s", "shuffle_read_bytes", "shuffle_write_bytes",
                "disk_bytes_spilled", "max_task_s", "task_count"):
        print(f"{key:22s} {delta(key)}")

baseline = {"wall_clock_s": 3600, "shuffle_read_bytes": 1_200_000_000_000,
            "shuffle_write_bytes": 1_200_000_000_000, "disk_bytes_spilled": 8_000_000_000,
            "max_task_s": 540, "task_count": 200_000}
tuned = {"wall_clock_s": 1_200, "shuffle_read_bytes": 20_000_000_000,
         "shuffle_write_bytes": 20_000_000_000, "disk_bytes_spilled": 0,
         "max_task_s": 95, "task_count": 40_000}

summarize(baseline, tuned)
print(json.dumps({"shuffle_reduction_pct": round(
    (1 - tuned["shuffle_read_bytes"] / baseline["shuffle_read_bytes"]) * 100, 1)}))

## Understand why shuffles are expensive

A shuffle redistributes rows so that all rows with the same key land in the same partition. `groupBy`,
`distinct`, `dropDuplicates`, non-broadcast joins, `repartition`, `orderBy` and windows with `partitionBy` all
shuffle. Each map task partitions its output by `hash(key) % numPartitions`, writes one block per reducer to
local disk, and reducers fetch their blocks from every mapper over the network: serialization, disk, network
and a sort or hash build.

**Where it matters**
- **Large aggregations and joins,** where shuffle bytes usually dominate runtime.
- **Unstable clusters:** a lost executor loses its shuffle files, and the map stage runs again.

**Watch out for**
- `orderBy` before a write is a range shuffle; drop it unless readers need sorted files.
- `repartition()` with no arguments is a full shuffle for no benefit.
- Moving 500 GB at an aggregate 1 GB/s costs at least 500 seconds before serialization and sorting.

### Example

Count the exchanges: a `groupBy` adds one, and an `orderBy` on top adds a range-partitioned second one.

In [ ]:
daily_revenue = big_orders.groupBy("order_date").agg(F.sum("amount").alias("revenue"))
ordered = daily_revenue.orderBy("order_date")

for name, frame in [("groupBy", daily_revenue), ("groupBy + orderBy", ordered)]:
    print(f"{name:<18} exchanges: {plan_of(frame, 'simple').count('Exchange ')}")

## Choose the number of shuffle partitions

`spark.sql.shuffle.partitions` (default 200) sets the reduce fan-out of every shuffle. Too few partitions
give large tasks that spill; too many give scheduling overhead and tiny output files. Derive it from a target
size of roughly 100 to 200 MB per partition: `partitions = ceil(shuffle_bytes / target_size)`.

With AQE on, the setting is only the starting count: `spark.sql.adaptive.advisoryPartitionSizeInBytes`
(64 MB by default) is the size AQE coalesces toward once it sees the real map output.

**Where it matters**
- **A 2 TB shuffle:** about 16,000 partitions at 128 MB, or 32,000 at 64 MB.
- **Tiny data in tests:** a handful of partitions instead of 200 empty tasks.

**Watch out for**
- A huge round number "just in case" with AQE off: tens of thousands of tiny tasks and files.
- Changing the setting after building a DataFrame applies to the next action, not to plans already executed.

### Example

With AQE off the aggregation keeps the configured 16 partitions; with AQE on it coalesces them. Then compute counts from bytes.

In [ ]:
by_customer = lambda: big_orders.groupBy("customer_id").agg(F.sum("amount").alias("revenue"))  # noqa: E731

spark.conf.set("spark.sql.adaptive.enabled", "false")
static_partitions = by_customer().rdd.getNumPartitions()

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
adaptive_partitions = by_customer().rdd.getNumPartitions()

print("AQE off:", static_partitions, "partitions | AQE on:", adaptive_partitions)

In [ ]:
import math


def shuffle_partitions_for(shuffle_bytes, target_bytes=128 * 1024**2):
    return math.ceil(shuffle_bytes / target_bytes)

for label, size in [("200 GB", 200 * 1024**3), ("2 TB", 2 * 1024**4)]:
    print(f"{label:>6} shuffle -> {shuffle_partitions_for(size):,} partitions at 128 MB")

## Let adaptive query execution re-plan

AQE (on by default since Spark 3.2) re-plans between query stages. When the map side of a shuffle finishes,
Spark knows the real partition sizes and can:

- coalesce many small post-shuffle partitions (`spark.sql.adaptive.coalescePartitions.enabled`);
- split skewed join partitions (`spark.sql.adaptive.skewJoin.enabled`); a partition counts as skewed above both
  `skewedPartitionThresholdInBytes` (256 MB) and `skewedPartitionFactor` (5) times the median;
- switch a sort-merge join to a broadcast join when a side turns out to be small;
- read shuffle files locally (`spark.sql.adaptive.localShuffleReader.enabled`).

**Where it matters**
- **Uneven shuffles:** 190 partitions of 5 MB and ten of 2 GB become a few 64 MB partitions plus split skewed ones.
- **Filters the optimizer cannot estimate,** which leave a side far smaller than planned.

**Watch out for**
- AQE acts only at shuffle boundaries. It cannot fix a skewed scan, a bad partition key, or a scan that reads every file.
- Reading the final plan needs a completed action; the example uses an internal API (`_jdf`) that Spark Connect does not have.

### Example

The optimizer's estimate rules out a broadcast; after running, AQE has switched to one because the filtered side is tiny.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(64 * 1024))  # 64 KB: too small for the raw scan

tiny_customers = big_customers.where("customer_id < 50")  # the planner cannot know this is tiny
joined = big_orders.join(tiny_customers.groupBy("customer_id").agg(F.max("customer_segment").alias("seg")), "customer_id")
before = plan_of(joined)
joined.collect()  # run it so AQE can re-plan with real sizes
final = joined._jdf.queryExecution().executedPlan().toString()  # the plan that actually ran (internal API)
print("initial plan:", "SortMergeJoin" if "SortMergeJoin" in before else "?",
      "| final plan:", "BroadcastHashJoin" if "BroadcastHashJoin" in final else "?")

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

## Fix data skew with salting

When a few keys hold most rows, one task owns most of the work. **Salting** adds a random bucket to the hot
side and replicates the small side across all buckets, so the hot key spreads over many tasks. Aggregations
salt in two phases: aggregate by `(key, salt)`, then by `key`. Use it when broadcasting is impossible and
AQE skew handling is not enough, and only after a key count confirmed the skew.

**Where it matters**
- **Guest checkouts, null or `UNKNOWN` keys,** and a few very large tenants.
- **Joins where both sides are too large to broadcast.**

**Watch out for**
- Salting without confirmed skew adds a shuffle for nothing.
- The small side grows by the salt factor: a 20 MB dimension salted 32 ways is 640 MB.
- A salt computed from a hash of the key alone does not help: every row of the hot key gets the same salt. Often the real fix is the source of the hot key (a null or placeholder).

### Example

Salt the join 16 ways and compare the largest partition; then salt an aggregation in two phases and check the totals match.

In [ ]:
salt_buckets = 16

orders_salted = big_orders.withColumn("salt", (F.rand(seed=42) * salt_buckets).cast("int"))
customers_salted = big_customers.crossJoin(spark.range(salt_buckets).withColumnRenamed("id", "salt"))

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force a shuffle join for the demo
plain = big_orders.join(big_customers, "customer_id")
salted = orders_salted.join(customers_salted, ["customer_id", "salt"]).drop("salt")

hot_before = max(partition_sizes(big_orders.repartition(16, "customer_id")))
hot_after = max(partition_sizes(orders_salted.repartition(16, "customer_id", "salt")))
print(f"largest join partition: {hot_before:,} rows before salting, {hot_after:,} after")

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

In [ ]:
partial = orders_salted.groupBy("customer_id", "salt").agg(F.sum("amount").alias("partial_amount"))
final = partial.groupBy("customer_id").agg(F.sum("partial_amount").alias("total_amount"))

direct = big_orders.groupBy("customer_id").agg(F.sum("amount").alias("total_amount"))
print("two-phase salted aggregate matches the direct aggregate")

## Tune joins and broadcasts

Start with the size and distribution of both sides: broadcast a small side, filter and select before joining,
deduplicate dimensions so facts are not multiplied, and handle hot keys.

- `BroadcastHashJoin` ships the small side to every executor; the large side never shuffles.
- `SortMergeJoin` shuffles and sorts both sides; it scales to two large inputs and spills gracefully.
- `ShuffledHashJoin` shuffles both sides but builds a hash table instead of sorting (chosen by AQE, or with
  `spark.sql.join.preferSortMergeJoin=false`).
- Non-equi joins fall back to `BroadcastNestedLoopJoin` or a Cartesian product: almost always a mistake.

Spark broadcasts automatically below `spark.sql.autoBroadcastJoinThreshold` (10 MB; `-1` disables it). At
runtime the small side is collected to the driver and then sent to executors, so the costs are driver memory,
executor memory and broadcast latency.

**Where it matters**
- **Fact-to-dimension joins,** where a broadcast removes the shuffle entirely.
- **A broadcast that stopped broadcasting** because a dimension grew past the threshold.

**Watch out for**
- Joining on an expression such as `lower(a.id) == lower(b.id)` instead of a clean key column.
- A dimension that is not unique on the key silently multiplies facts.
- Raising the threshold globally to make one join broadcast. Hint that join, after checking its size.

### Example

A non-unique dimension inflates the join; then the plan switches from sort-merge to broadcast when the small side is hinted.

In [ ]:
dim_with_history = big_customers.unionByName(big_customers.where("customer_id < 10"))  # two versions of 10 rows

fanned_out = big_orders.join(dim_with_history, "customer_id").count()
deduped = big_orders.join(dim_with_history.dropDuplicates(["customer_id"]), "customer_id").count()
print(f"rows after join: {fanned_out:,} with a non-unique dimension, {deduped:,} after dedupe")

In [ ]:
from pyspark.sql.functions import broadcast

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
shuffled_plan = plan_of(big_orders.join(big_customers, "customer_id"))
hinted_plan = plan_of(big_orders.join(broadcast(big_customers), "customer_id"))
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

print("threshold off -> SortMergeJoin; broadcast() hint -> BroadcastHashJoin")

## Repartition, coalesce and partition by column

`repartition(n)` is a full shuffle (round-robin, or hash when you pass columns); it can raise or lower the
count and balances by construction. `coalesce(n)` merges existing partitions without a shuffle, can only lower
the count, and keeps any imbalance because it never splits a partition.

`repartition(n, col)` hash-partitions so equal keys always meet, which is what a keyed join, aggregation or
dedup needs. Before a partitioned write, `repartition("order_date")` followed by `partitionBy("order_date")`
writes about one file per date; without it, each task writes one file per date it holds.

**Where it matters**
- **After a selective filter:** 1,000 sparse partitions down to a sensible count.
- **Before a partitioned write,** to control the number of files per partition.

**Watch out for**
- `coalesce(1)` on a large result funnels the whole write onto one core.
- All nulls hash to one partition, and one hot value makes one partition huge whatever `n` is.
- Repartitioning by one key and then shuffling on another: two exchanges where one was needed.

### Example

Compare partition sizes after hashing on a skewed key, coalescing and repartitioning; then count files written with and without `repartition` by date.

In [ ]:
skewed = big_orders.repartition(8, "customer_id")  # the hot key lands in a single partition
merged = skewed.coalesce(4)  # narrow: merges whole partitions, keeps the hot one intact
balanced = skewed.repartition(4)  # full round-robin shuffle

for name, frame in [("hash by customer", skewed), ("coalesce(4)", merged), ("repartition(4)", balanced)]:
    print(f"{name:<17} {partition_sizes(frame)}")

exchanges = lambda df: plan_of(df, "simple").count("Exchange ")  # noqa: E731

In [ ]:
def file_count(path):
    return sum(name.endswith(".parquet") for _, _, names in os.walk(path) for name in names)

scattered_path = os.path.join(WORK_DIR, "scattered")
clustered_path = os.path.join(WORK_DIR, "clustered")
big_orders.repartition(8).write.mode("overwrite").partitionBy("order_date").parquet(scattered_path)
big_orders.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(clustered_path)

print("files without repartition by date:", file_count(scattered_path))
print("files with repartition by date   :", file_count(clustered_path))

## Avoid small files

Spark packs files into scan tasks using `spark.sql.files.maxPartitionBytes` (128 MB) and
`spark.sql.files.openCostInBytes` (4 MB, the modelled cost of opening a file). With many tiny files, listing,
opening and footer reads dominate, and table formats carry more per-file metadata. Control the file count
when writing, cap rows per file with `maxRecordsPerFile`, and compact what already exists with the table
format:

```sql
-- Delta
OPTIMIZE main.silver.orders WHERE order_date >= date '2026-10-01';
-- Iceberg (Spark procedure)
CALL catalog.system.rewrite_data_files(table => 'silver.orders');
```

**Where it matters**
- **Streaming sinks** that write a few files every minute.
- **Over-partitioned tables:** 500,000 files of 1 MB compacted to 1 GB files leaves about 500.

**Watch out for**
- Rewriting a whole table to fix one partition.
- Leaving write-time compaction (`delta.autoOptimize.optimizeWrite`, `autoCompact`) off on streaming tables.

### Example

Write one day as 24 files, then compact it into one (fine here only because the day is tiny).

In [ ]:
small_path = os.path.join(WORK_DIR, "small_files")
compact_path = os.path.join(WORK_DIR, "compacted")
day = big_orders.where("order_date = date '2026-10-05'")

day.repartition(24).write.mode("overwrite").parquet(small_path)
spark.read.parquet(small_path).coalesce(1).write.mode("overwrite").parquet(compact_path)

print("files before compaction:", file_count(small_path), "| after:", file_count(compact_path))

## Prune partitions and push down predicates

**Partition pruning** uses the directory layout: Spark evaluates the predicate on partition values and lists
the survivors under `PartitionFilters`. **Pushdown and data skipping** work inside files: Parquet row-group
min/max values and Delta per-file statistics let the reader skip data, shown under `PushedFilters`. Wrapping a
regular column in a function hides the literal from the reader.

**Dynamic partition pruning** (DPP) applies the same idea at runtime: when a partitioned fact table is joined
to a filtered dimension, Spark first computes the dimension's matching keys and uses them to prune the
fact table's partitions.

Catalyst pushes deterministic filters and projections toward the scan, but not through aggregations,
windows, `distinct`, Python UDFs or non-deterministic expressions, so write the filter early anyway.

**Where it matters**
- **Date-partitioned facts:** a one-day query over five years reads about 1/1825 of the files.
- **Star-schema queries** that filter a calendar or store dimension and join it to a partitioned fact.

**Watch out for**
- `lower(country) = 'us'`, `cast(ts as string) like '2026-10%'` or comparing a timestamp with a string literal defeat pushdown on a regular column.
- Delta collects statistics for the first 32 columns by default (`delta.dataSkippingNumIndexedCols`); put filter columns early.

### Example

A raw predicate is pushed down; a wrapped one is not; an expression on the partition column still prunes. Then Catalyst moves a late filter down to the same scan.

In [ ]:
def plan_line(df, prefix):
    return next(line.strip() for line in plan_of(df).splitlines() if line.strip().startswith(prefix))

raw_predicate = big_orders.where("status = 'paid'")
wrapped_predicate = big_orders.where("lower(status) = 'paid'")
partition_expression = big_orders.where("date_format(order_date, 'yyyy-MM-dd') = '2026-10-05'")

print(plan_line(raw_predicate, "PushedFilters"))
print(plan_line(wrapped_predicate, "PushedFilters"))
print(plan_line(partition_expression, "PartitionFilters"))

In [ ]:
early = (
    big_orders.where("order_date >= date '2026-10-20'")
    .select("customer_id", "amount")
    .join(big_customers, "customer_id")
    .groupBy("customer_segment").agg(F.sum("amount").alias("revenue"))
)
late = (
    big_orders.join(big_customers, "customer_id")
    .where("order_date >= date '2026-10-20'")
    .groupBy("customer_segment").agg(F.sum("amount").alias("revenue"))
)
print("Catalyst pushed the late filter down to the same scan:", plan_line(late, "PartitionFilters"))

### Example: dynamic partition pruning

Join the date-partitioned orders to a calendar filtered to one holiday. The plan prunes order partitions with a dynamic pruning expression.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
calendar = spark.createDataFrame(
    [(date(2026, 10, 5), "holiday"), (date(2026, 10, 6), "normal")], "order_date date, day_type string"
)
holiday_orders = big_orders.join(calendar.where("day_type = 'holiday'"), "order_date")

print(plan_line(holiday_orders, "PartitionFilters"))
print("orders on the holiday:", holiday_orders.count(), "of", big_orders.count())

**What to notice:** The order scan's `PartitionFilters` contains a `dynamicpruningexpression`: at runtime Spark computes which
`order_date` values survive the calendar filter and reads only those partitions, one day of thirty. The
query never names a date on `big_orders` itself.

## Bucket tables and keep statistics current

**Bucketing** (`bucketBy(n, col)` with `saveAsTable`) hash-partitions a table into a fixed number of files
by a key when it is written. Two tables bucketed the same way on the join key can be joined without a
shuffle, because matching keys are already in matching buckets. It pays off for large tables joined on the
same key over and over.

The planner chooses join strategies from **statistics**. `ANALYZE TABLE ... COMPUTE STATISTICS` records row
counts and sizes (and, `FOR COLUMNS`, distinct counts and min/max values) in the catalog; the cost-based
optimizer (`spark.sql.cbo.enabled`) uses column statistics to order joins and estimate filters.

**Where it matters**
- **Repeated large joins** on the same key, such as orders to order lines by `order_id`.
- **Wrong join choices** caused by missing or stale statistics on catalog tables.

**Watch out for**
- Bucketing only helps when both sides use the same key and compatible bucket counts, and readers go through the catalog table.
- Each write task can create one file per bucket, so bucketing large writes multiplies file counts.
- Delta and Iceberg prefer clustering (liquid clustering, Z-order, sort orders) over Hive-style buckets.
- Statistics go stale after large loads; refresh them as part of the load.

### Example

Bucket both tables by `customer_id` into 8 buckets. With broadcasting off, their join plans no exchange at
all. Then compute statistics and read them back from the catalog.

In [ ]:
big_orders.write.bucketBy(8, "customer_id").sortBy("customer_id").mode("overwrite").saveAsTable("orders_bucketed")
big_customers.write.bucketBy(8, "customer_id").sortBy("customer_id").mode("overwrite").saveAsTable("customers_bucketed")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
bucketed_join = spark.table("orders_bucketed").join(spark.table("customers_bucketed"), "customer_id")
plain_join = big_orders.join(big_customers, "customer_id")
bucketed_exchanges = plan_of(bucketed_join, "simple").count("Exchange ")
plain_exchanges = plan_of(plain_join, "simple").count("Exchange ")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
print("exchanges - bucketed:", bucketed_exchanges, "| plain:", plain_exchanges)

spark.sql("ANALYZE TABLE customers_bucketed COMPUTE STATISTICS FOR COLUMNS customer_id")
stats = spark.sql("DESCRIBE EXTENDED customers_bucketed").where("col_name = 'Statistics'").first()
print(stats.data_type)

**What to notice:** With broadcasting off, the plain join shuffles both sides (two exchanges). The bucketed tables already
have matching keys in matching buckets, so their join plans no exchange at all. `ANALYZE TABLE` stored the
row count and size in the catalog, where the planner reads them.

## Cache, persist and checkpoint

Cache only a DataFrame that is reused, expensive to recompute, and fits. `cache()` and `persist()` are lazy:
blocks are stored on the first action. `MEMORY_ONLY` keeps deserialized objects (fast, large; evicted
partitions are recomputed), `MEMORY_AND_DISK` spills evicted blocks, and `DISK_ONLY` always writes to disk.
Storage shares the unified memory region with execution, so a large shuffle can evict cached blocks.

A **checkpoint** is different: `df.checkpoint()` writes the data to reliable storage (the directory set with
`setCheckpointDir`) and **cuts the lineage**, so later plans start from the saved data. `localCheckpoint()`
writes to executor storage instead: faster, but lost with the executor. Use them for very long plans, such
as iterative loops, where analysis time or recomputation after a failure becomes the problem.

**Where it matters**
- **Caching:** a cleaned frame read by both a valid-rows write and a quarantine write.
- **Checkpointing:** an iterative algorithm or a loop that adds columns hundreds of times.

**Watch out for**
- 100 GB cached on 20 executors with 16 GB heaps does not fit: the unified region is 0.6 of (heap - 300 MB), about 188 GB, and storage keeps only half of it safe from eviction.
- Caching by reflex, caching a frame read once, and forgetting `unpersist()`.
- A checkpoint is a full write of the data; it is not free.

### Example

Persist a filtered frame that two actions reuse, then checkpoint a frame built in a loop and check that its plan no longer reaches the Parquet scan.

In [ ]:
from pyspark import StorageLevel

base = (
    big_orders
    .where("order_date >= date '2026-10-20'")
    .select("order_id", "customer_id", "amount", "order_date")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
high_value = base.where("amount >= 90").count()  # the first action materializes the cache
daily = base.groupBy("order_date").agg(F.sum("amount").alias("revenue")).count()

print("storage level:", base.storageLevel, "| high-value big_orders:", high_value, "| days:", daily)
base.unpersist()

In [ ]:
spark.sparkContext.setCheckpointDir(os.path.join(WORK_DIR, "checkpoints"))

looped = big_orders.select("order_id", "amount")
for step in range(5):  # stands in for an iterative job whose plan keeps growing
    looped = looped.withColumn(f"amount_{step}", F.col("amount") + step)

saved = looped.checkpoint()  # eager by default: writes the data and cuts the lineage
print("--- before the checkpoint ---")
looped.explain()
print("--- after the checkpoint ---")
saved.explain()

**What to notice:** Before the checkpoint, the plan still starts from `Scan parquet` and repeats every projection. After it, the
plan starts from `Scan ExistingRDD`: the saved data. A failure or a further loop no longer replays the
lineage.

## Avoid driver memory problems

The driver holds the plan, scheduler state, broadcast values and every collected result. `collect()` and
`toPandas()` pull the whole result into one JVM; `spark.driver.maxResultSize` (1 GB) caps it, and the driver
heap often fails first. A 50-million-row result at 200 bytes per row is 10 GB. Aggregate in the cluster,
write large results to tables, and collect only bounded samples.

**Where it matters**
- **Notebooks and reports** that pull data into pandas for plotting.
- **Large broadcasts,** which are built on the driver first.

**Watch out for**
- `toPandas()` on an unfiltered table.
- Raising `maxResultSize` instead of reducing the result.

### Example

Aggregate in the cluster and bring back one row, and take a bounded sample.

In [ ]:
summary = big_orders.agg(F.sum("amount").alias("total"), F.count("*").alias("rows")).first()
sample = big_orders.select("order_id", "customer_id").limit(5).collect()

print("aggregated in the cluster:", summary.asDict())
print("bounded sample:", [row["order_id"] for row in sample])

## Size executors, memory and parallelism

Three independent numbers: cores per executor (task slots), memory per executor (shared by those tasks), and
executor count. Total parallelism is `executors x cores`, and shuffle partitions should be a small multiple of
it so the last wave does not leave cores idle.

An executor heap loses 300 MB of reserved memory; the unified region is `spark.memory.fraction` (0.6) of the
rest, shared by execution (shuffles, sorts, joins) and storage (cache), with `spark.memory.storageFraction`
(0.5) protected for storage. `spark.executor.memoryOverhead` (default `max(384 MB, 0.10 x heap)`) covers Python
workers, network buffers and JVM internals, and the container is heap plus overhead.

```python
spark = (
    SparkSession.builder
    .appName("large-orders-aggregation")
    .config("spark.executor.instances", "60")
    .config("spark.executor.cores", "5")
    .config("spark.executor.memory", "38g")
    .config("spark.executor.memoryOverhead", "4g")
    .config("spark.sql.shuffle.partitions", "1200")
    .getOrCreate()
)
```

**Where it matters**
- **Cluster sizing:** 10 nodes with 32 cores and 256 GB each; leave a core and 1 GB per node for the OS; five cores per executor gives 6 executors per node, 60 in total, about 42 GB each, roughly 38 GB of heap after overhead.
- **Shared clusters:** dynamic allocation (`spark.dynamicAllocation.*` with shuffle tracking) suits bursty jobs; static allocation suits stable nightly jobs.

**Watch out for**
- Very wide executors create long GC pauses; many one-core executors multiply JVM overhead and broadcast copies. Three to five cores and 8 to 32 GB is the usual landing zone.
- Containers killed by the cluster manager usually mean overhead memory, especially for Python-heavy jobs.
- These are launch-time settings; `spark.conf.set` cannot change them in a running session.

### Example

Compute the container size, the unified memory region and the execution memory per task for a 16 GB, five-core executor.

In [ ]:
def executor_layout(heap_gb, cores, memory_fraction=0.6, reserved_gb=0.3):
    overhead_gb = max(0.384, 0.10 * heap_gb)
    unified_gb = memory_fraction * (heap_gb - reserved_gb)
    return {
        "container_gb": round(heap_gb + overhead_gb, 1),
        "unified_gb": round(unified_gb, 1),
        "execution_per_task_gb": round(unified_gb / cores, 2),
    }

layout = executor_layout(heap_gb=16, cores=5)
print(layout)

## Handle spill, serialization and GC

When a sort, hash aggregate or hash join exhausts its task's execution memory, Spark writes sorted runs to
local disk and merges them later: correct, but disk-bound. Spill is per task, so spill on a handful of tasks
is itself a skew signal. The cause is almost always partition size versus per-task memory: a 3 GB partition
with about 1 GB per task spills.

Fix spill in this order: more shuffle partitions or a smaller advisory size; filter and aggregate earlier;
drop an unnecessary sort; broadcast the small side; only then add executor memory.

DataFrames already use Tungsten's compact binary format. Kryo serialization
(`spark.serializer=org.apache.spark.serializer.KryoSerializer`) matters for RDD shuffles, broadcasts and
caches of objects. High GC time means too many live objects: oversized caches, very wide rows or giant
executors. G1 (`-XX:+UseG1GC` in `spark.executor.extraJavaOptions`) is the usual collector.

**Where it matters**
- **Stages that spill** on large joins and aggregations.
- **RDD-heavy or cache-heavy jobs** with high GC time.

**Watch out for**
- If one partition holds 10 percent of a 2 TB shuffle, no amount of memory helps: the partition is the problem.
- Caching more while already spilling makes it worse.
- Expecting a serializer change to speed up an I/O-bound scan or a skewed task.

### Example

A stage shuffles 2 TB into 200 partitions: about 10 GB each, against roughly 1.9 GB of execution memory per
task on the executor above. Every task spills. At a 64 MB advisory size AQE targets about 32,000 partitions,
each well inside task memory, and spill drops to zero without adding memory:

```python
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
spark.conf.set("spark.sql.shuffle.partitions", "32000")  # the starting count AQE coalesces from
```

## Tune Python UDFs and pandas UDFs

A row-at-a-time Python UDF serializes every row to a Python worker and back, is opaque to Catalyst, and breaks
whole-stage code generation. Order of preference: built-in expression, SQL or higher-order function,
vectorized pandas UDF, row UDF. Pandas UDFs amortize the boundary with Arrow batches
(`spark.sql.execution.arrow.maxRecordsPerBatch`, 10,000 rows by default); `mapInPandas` can change the row
count and `applyInPandas` runs once per group.

**Where it matters**
- **Hot paths** that call a UDF on every row of a large table.
- **ML scoring,** where a pandas UDF over a model is the right tool.

**Watch out for**
- On Databricks, Photon accelerates native operators but not Python UDFs, so a UDF in the hot path loses the speedup.
- A pandas UDF still crosses the JVM boundary; filter and aggregate in the JVM first.

### Example

The same tier rule as a built-in and as a UDF: same answer, but only the UDF plan leaves the JVM.

In [ ]:
from pyspark.sql import types as T


@F.udf(T.StringType())
def tier_udf(amount):
    return "high" if amount is not None and amount >= 90 else "standard"

builtin_tier = big_orders.withColumn("tier", F.when(F.col("amount") >= 90, "high").otherwise("standard"))
udf_tier = big_orders.withColumn("tier", tier_udf("amount"))

print("same result; only the UDF plan leaves the JVM")

## Tune Structured Streaming and table layout

A streaming query is a micro-batch loop: each trigger runs a normal job over newly available data. Bound each
batch with `maxOffsetsPerTrigger` (Kafka) or `maxFilesPerTrigger` (files), choose a trigger interval each batch
can finish within, bound state with watermarks, and keep one stable checkpoint per stream.

Layout changes the cost of every future query. Aim for 128 MB to 1 GB files; `OPTIMIZE` compacts small files,
`ZORDER BY (col)` co-locates values for data skipping, and **liquid clustering** (`CLUSTER BY (cols)`) records
clustering keys in metadata and re-clusters incrementally. Liquid clustering cannot be combined with
`partitionBy` or `ZORDER`. `VACUUM` removes unreferenced files older than the retention (7 days by default);
it saves storage, not time, and ends time travel for that window.

```sql
OPTIMIZE main.silver.orders WHERE order_date >= date '2026-10-01' ZORDER BY (customer_id);
ALTER TABLE main.silver.orders CLUSTER BY (order_date, customer_id);
VACUUM main.silver.orders RETAIN 168 HOURS;
```

**Where it matters**
- **Streams whose latency keeps climbing** as the source grows.
- **Tables read by many queries,** where layout work pays back on every read.

**Watch out for**
- Stateful dedup without a watermark: state grows until the job slows down a week later.
- A full-table scan inside `foreachBatch` that repeats every micro-batch.
- Optimizing on a schedule instead of on evidence, and partitioning by a high-cardinality column.

### Example

With a 60-second trigger, batches start taking 90 seconds as a topic grows, so each batch starts later with
more data and latency climbs without bound. Capping `maxOffsetsPerTrigger` and widening the trigger to two
minutes stabilizes it:

```python
(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("maxOffsetsPerTrigger", 500_000)
    .load()
    .writeStream
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="2 minutes")
    .toTable("main.silver.orders")
)
```

## What changes between Spark 3 and Spark 4

Most tuning advice carries over unchanged. These are the differences that show up in jobs and interviews:

- **ANSI SQL mode** is on by default in Spark 4.0 (`spark.sql.ansi.enabled=true`): invalid casts, overflow and
  division by zero raise errors instead of returning null. Jobs that silently produced nulls on Spark 3 can
  fail after an upgrade; that is a correctness fix.
- **AQE** has been on by default since 3.2. Start from AQE and tune `advisoryPartitionSizeInBytes` before
  hand-setting shuffle partitions per query.
- **Event logs** roll by default, so each application writes a directory of `events_*` files. The
  `event_log_files` helper above handles both layouts.
- **Runtime:** Spark 4 requires Java 17 or 21 and Scala 2.13. Old Java 8 GC flags do not carry over.
- **Spark Connect** is a first-class option. A remote client has no `df._jdf` or `sparkContext`, so plan
  inspection goes through `explain` and the UI.
- **Python:** PySpark 4.0 needs Python 3.9 or newer, and pandas UDFs need `pandas` and `pyarrow` on every
  worker.

Read each release's migration guide before upgrading a cluster; behaviour changes are listed there.

## Tuning exercises

Standard-library models of the quantities you read from the UI and `explain`: shuffle bytes, broadcast
thresholds, partition row counts and executor arithmetic. Work out each answer before running the cell; the
asserts check the formula, not a real cluster.

### Try it 1. Choose shuffle partitions

**Task.** A shuffle moves 640 GB (binary units). Compute the number of shuffle partitions needed for a 128 MB target partition size, then for a 256 MB target.

**Hint.** `partitions = ceil(shuffle_bytes / target_bytes)`. One GB here is `1024` MB.

**Expected result.** `5,120` partitions for 128 MB and `2,560` for 256 MB.

In [ ]:
import math


def shuffle_partitions(shuffle_gb, target_mb):
    shuffle_mb = shuffle_gb * 1024
    return math.ceil(shuffle_mb / target_mb)

partitions_128 = shuffle_partitions(640, 128)
partitions_256 = shuffle_partitions(640, 256)
print(f"640 GB at 128 MB/partition -> {partitions_128:,} partitions")
print(f"640 GB at 256 MB/partition -> {partitions_256:,} partitions")

### Try it 2. Broadcast threshold decision

**Task.** Using the default auto-broadcast threshold of 10 MB, decide whether a 7.5 MB dimension and a 45 MB dimension would be broadcast. Then show the driver and network cost of forcing the 45 MB dimension to broadcast across 60 executors.

**Hint.** Compare bytes against `10 * 1024 * 1024`. A forced broadcast replicates the relation to every executor.

**Expected result.** The 7.5 MB dimension broadcasts, the 45 MB one does not, and forcing it moves about `2,700` MB in total.

In [ ]:
MB = 1024 * 1024
THRESHOLD = 10 * MB
EXECUTORS = 60


def would_broadcast(size_mb):
    return size_mb * MB <= THRESHOLD

small, large = 7.5, 45.0
print(f"{small} MB -> broadcast: {would_broadcast(small)}")
print(f"{large} MB -> broadcast: {would_broadcast(large)}")

network_mb = large * EXECUTORS
print(f"forced broadcast network = {network_mb:,.0f} MB")

### Try it 3. Detect skew from per-partition row counts

**Task.** Given the row counts of eight shuffle partitions, compute the median, the maximum, and the ratio of max to median. Flag the stage as skewed when the max exceeds five times the median.

**Hint.** Sort the counts and take the middle value, averaging the two middle values for an even count.

**Expected result.** Median `101`, maximum `5,000`, ratio about `49.5`, and the stage is skewed.

In [ ]:
counts = [100, 105, 98, 102, 100, 5000, 99, 103]


def median(values):
    ordered = sorted(values)
    mid = len(ordered) // 2
    if len(ordered) % 2 == 1:
        return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2

med = median(counts)
peak = max(counts)
ratio = peak / med
skewed = ratio > 5

print(f"median = {med:.1f}, max = {peak}, ratio = {ratio:.1f}")
print("skewed:", skewed)

### Try it 4. Salt a hot key and recompute the distribution

**Task.** One million rows are keyed by customer. The key `guest` holds 40 percent and four normal keys split the rest evenly. Compute the largest partition share before salting, then salt `guest` into 32 buckets evenly and recompute it.

**Hint.** After salting, the largest partition is either a salted bucket of the hot key or one of the normal keys, whichever is larger.

**Expected result.** The largest share falls from `0.40` to `0.15`, roughly a `2.67x` improvement.

In [ ]:
TOTAL = 1_000_000
SALT_BUCKETS = 32
hot = int(TOTAL * 0.40)
normal = (TOTAL - hot) / 4

before = hot / TOTAL
salted_bucket = hot / SALT_BUCKETS
after = max(salted_bucket, normal) / TOTAL

print(f"before: hot key share = {before:.2f}")
print(f"after : largest share = {after:.2f} (salted bucket = {salted_bucket:,.0f} rows)")
print(f"improvement = {before / after:.2f}x")

### Try it 5. Decide repartition versus coalesce

**Task.** After a filter, a DataFrame has 1,000 partitions holding 4 GB in total and the downstream step wants 16 tasks. Decide between `coalesce(16)` and `repartition(16)` using a 128 MB target partition size. Repeat for a 512 MB result with 4 target tasks.

**Hint.** `coalesce` merges whole upstream partitions without a shuffle, so it is safe only when the merged partition fits the target. Otherwise you need `repartition`.

**Expected result.** 4 GB into 16 tasks is 256 MB each, so use `repartition`; 512 MB into 4 tasks is 128 MB each, so `coalesce` is fine.

In [ ]:
MB = 1024 * 1024


def choose_op(total_mb, target_tasks, target_mb=128):
    per_task = total_mb / target_tasks
    choice = "coalesce" if per_task <= target_mb else "repartition"
    return choice, per_task

choice_a, per_a = choose_op(4 * 1024, 16)
choice_b, per_b = choose_op(512, 4)
print(f"4096 MB / 16 = {per_a:.0f} MB per task -> {choice_a}")
print(f" 512 MB /  4 = {per_b:.0f} MB per task -> {choice_b}")

### Try it 6. Estimate executor count from cores and memory

**Task.** A cluster has 10 nodes with 32 cores and 256 GB of RAM each. Leave one core and 1 GB per node for the operating system and the NodeManager. With 5 cores per executor, compute executors per node, total executors, total cores used, and the memory per executor.

**Hint.** `executors_per_node = floor((cores_per_node - 1) / cores_per_executor)`, then divide the usable memory by that count.

**Expected result.** `6` executors per node, `60` in total, `30` cores used per node, and about `42` GB of memory each.

In [ ]:
NODES = 10
CORES_PER_NODE = 32
MEM_PER_NODE_GB = 256
CORES_PER_EXECUTOR = 5

usable_cores = CORES_PER_NODE - 1
usable_mem = MEM_PER_NODE_GB - 1

executors_per_node = usable_cores // CORES_PER_EXECUTOR
cores_used = executors_per_node * CORES_PER_EXECUTOR
mem_per_executor = usable_mem // executors_per_node
total_executors = executors_per_node * NODES

print(f"executors per node = {executors_per_node}")
print(f"total executors    = {total_executors}")
print(f"cores used per node= {cores_used} of {CORES_PER_NODE}")
print(f"memory per executor= {mem_per_executor} GB")

### Try it 7. Compute AQE coalescing of small partitions

**Task.** A shuffle produces 200 partitions of 2 MB each. AQE coalesces adjacent partitions toward `spark.sql.adaptive.advisoryPartitionSizeInBytes = 64m`. Compute the number of partitions after coalescing and the resulting average partition size.

**Hint.** `coalesced = ceil(total_bytes / advisory_bytes)`. Each coalesced partition packs whole input partitions, so the total is unchanged.

**Expected result.** `400` MB coalesces into `7` partitions of about `57.1` MB each.

In [ ]:
import math

INPUT_PARTITIONS = 200
PARTITION_MB = 2
ADVISORY_MB = 64

total_mb = INPUT_PARTITIONS * PARTITION_MB
coalesced = math.ceil(total_mb / ADVISORY_MB)
avg_mb = total_mb / coalesced

print(f"total = {total_mb} MB across {INPUT_PARTITIONS} partitions")
print(f"coalesced -> {coalesced} partitions, average {avg_mb:.1f} MB")

### Try it 8. Compare two plans by estimated shuffle bytes

**Task.** Plan A is a sort-merge join that shuffles 1.2 TB in each direction. Plan B broadcasts a 300 MB dimension to 60 executors and shuffles the fact side not at all. Compare total bytes moved across the network and compute the reduction factor.

**Hint.** Plan A moves its shuffle write plus its shuffle read. Plan B moves the broadcast size once per executor. Convert everything to MB before comparing.

**Expected result.** Plan A moves `2.4` TiB, Plan B about `18,000` MB, a reduction of roughly `139.8x`.

In [ ]:
MB_PER_GIB = 1024

plan_a_write_mb = 1.2 * 1024 * MB_PER_GIB
plan_a_read_mb = 1.2 * 1024 * MB_PER_GIB
plan_a_total = plan_a_write_mb + plan_a_read_mb

broadcast_mb = 300
executors = 60
plan_b_total = broadcast_mb * executors

reduction = plan_a_total / plan_b_total
print(f"plan A moves {plan_a_total:,.0f} MB")
print(f"plan B moves {plan_b_total:,.0f} MB")
print(f"reduction = {reduction:.1f}x")

## Real-world tuning cases

### 1. A nightly job that regressed from 20 minutes to 3 hours

One tenant's data grew; stage 7 showed one task at 90 percent of the stage time plus spill -
skew plus a fixed, oversized partition count. Fix: enable AQE coalescing and skew join with a
128 MB advisory size. Runtime returned to minutes with no memory change.

### 2. A join on a hot customer key

One customer produced 40 percent of rows and AQE skew handling was not enough, so the join was
salted into 16 buckets (the salting section above). Measure the key distribution first - salting
adds a shuffle.

### 3. Small files after streaming ingestion

A streaming sink wrote thousands of tiny files per hour, and planning and file opens dominated
scans. Fix: compact with the table format (`OPTIMIZE`, `rewrite_data_files`) and cap output with
`spark.sql.files.maxRecordsPerFile` or a repartition before writing.

### 4. A broadcast join that stopped broadcasting

A "small" product dimension grew past 10 MB, so every run shuffled both sides. Fix: a deliberate
`broadcast()` hint or a higher threshold, after checking driver and executor memory.

### 5. Streaming latency that kept climbing

Each micro-batch consumed everything available and grew larger. Fix: `maxOffsetsPerTrigger` plus
a 2-minute trigger, and a watermark on stateful operators.

### 6. Driver OOM from collecting results

`collect()` and `toPandas()` on large frames killed the driver. Fix: aggregate in the cluster and
`limit()` any sample that must cross to the driver.

In every case the UI or the plan named the bottleneck before anyone changed a setting, and the
proof was a byte count or a task duration. In an interview give the numbers and the mechanism:
"stage 7 had one task at 90 percent of the stage time and spill; AQE split the skewed partition,
spill went to zero, and runtime fell from three hours to twenty minutes" - not "I enabled AQE".

## What tuning checklist should you use in interviews?

+ Inspect the Spark UI and the physical plan first.
+ Check input size, file count, and partition pruning.
+ Filter rows and select columns early; avoid Python UDFs.
+ Broadcast small dimensions, deduplicate dimensions, handle verified skew (AQE, then salting).
+ Size shuffle partitions and enable AQE.
+ `repartition` to balance, `coalesce` to reduce output partitions.
+ Cache only reused, expensive frames; never `collect()` large data.
+ Compact small files and write sensible file sizes.
+ Validate with runtime, shuffle and spill metrics, and output correctness.

Strong answer: "I would not start with executor memory. I would read the plan and the UI, decide
whether the bottleneck is scan, shuffle, skew, join, spill, or output files, apply the smallest
targeted fix, and measure again." Naming a config is much weaker than naming the evidence that
justifies it.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** One task out of 200 takes 12 minutes while the median takes 20 seconds. What is it, and what do you try first?

**A.** Skew: one partition holds far more data. Enable AQE skew join first; if that is not enough, salt or isolate the hot key, or fix its source (often a null or placeholder key). More executors will not help.

**Q2.** How do you choose `spark.sql.shuffle.partitions`?

**A.** From bytes: `ceil(shuffle_bytes / target)` with a target of about 100 to 200 MB. With AQE on, set a generous starting count and let `advisoryPartitionSizeInBytes` coalesce.

**Q3.** What is the difference between `repartition` and `coalesce`?

**A.** `repartition` is a full shuffle that can raise or lower the count and balances partitions. `coalesce` merges partitions without a shuffle, can only lower the count, and keeps imbalance.

**Q4.** When does dynamic partition pruning apply?

**A.** When a partitioned table is joined on its partition column to another side that is filtered. Spark computes the matching keys at runtime and prunes the partitioned side's files.

**Q5.** How is `checkpoint()` different from `cache()`?

**A.** `cache` keeps data for reuse but keeps the lineage. `checkpoint` writes the data to reliable storage and cuts the lineage, so later plans start from the saved data.

**Q6.** A stage spills heavily. Why is adding executor memory the last step?

**A.** Spill comes from partitions larger than per-task memory. More partitions, earlier filtering, or a broadcast usually remove it at no cost; memory is the expensive lever.

**Q7.** Why does a broadcast join sometimes fail instead of running slowly?

**A.** The small side is collected on the driver and copied to every executor. If it is not actually small, the driver or executors run out of memory.

## Key takeaways

- Fix the plan before you change configuration: read `explain`, the UI and the event log first.
- Measure before and after on the same data, one change at a time, and check the output is identical.
- One slow task out of 200 is skew, not a slow cluster. Try AQE skew join, then salt or isolate the hot key.
- Size shuffle partitions from bytes, broadcast only what is genuinely small, and write files of 128 MB to 1 GB.
- Prune partitions (statically and dynamically), bucket only for repeated large joins, and keep statistics current.
- Cache only frames several actions reuse; checkpoint to cut long lineages; add memory only when spill or GC says so.

**Next:** [PySpark 8: real-world use cases and interviews](pyspark-08-real-world-and-interviews.ipynb)